# 1D TFIM — HVA Experiment (GNN-prefilled)

Self-contained notebook to construct the **Transverse-Field Ising Model (TFIM)** in 1D and the
**Hamiltonian Variational Ansatz (HVA)** circuits used by the pipeline, for **N = 10, 20, 30 qubits**.

The notebook is organized in two layers:

- **Layer 1 (standalone):** Hamiltonian + HVA circuit + exact references + a full **reconstruction
  recipe**. Depends only on `numpy`, `qiskit`, `scipy`. This is the hand-off package for the
  ED / DMRG teams — it runs anywhere without installing the project, and it contains everything
  needed to rebuild the *filled* circuit from the saved angles alone.
- **Layer 2 (integrated, `USE_GNN`):** loads the project's GNN predictor, predicts the HVA angles
  for `chain_1d`, **fills the circuit**, evaluates $\langle H\rangle$ vs the exact $E_0$ (N=10, 20, 30),
  computes ground-state **fidelity for N=10** with the project's best method, and exports images.
  Requires `qmbp_simulation` + `torch` + a trained checkpoint.

**Sweep parameters:** `h in {1.0, 1.2, 1.4}` (near/above the 1D critical point `h_c = J = 1`),
`J = 1.0`, `p_layers = 1`, open boundary conditions.

**Exact references:** N=10 dense ED, N=20 sparse Lanczos, N=30 DMRG (TeNPy).

## 1. The Hamiltonian (full description)

The 1D transverse-field Ising model on an **open chain** of $N$ qubits:

$$
H \;=\; -\,J \sum_{i=0}^{N-2} Z_i Z_{i+1} \;-\; h \sum_{i=0}^{N-1} X_i
$$

**Conventions (must be matched by ED / DMRG references):**

| Item | Value |
|------|-------|
| Coupling sign | **Ferromagnetic**: interaction term is $-J\,Z_iZ_{i+1}$ with $J=1$ |
| Field sign | Transverse field term is $-h\,X_i$ |
| Boundary conditions | **Open** (no $Z_{N-1}Z_0$ term) |
| Edges | $(i, i{+}1)$ for $i = 0 \dots N-2$ → exactly $N-1$ bonds |
| Qubit indexing | site $i$ ↔ qubit $i$; Qiskit little-endian (qubit 0 = rightmost tensor factor) |
| Critical point | $h_c = J = 1$ (1D TFIM, thermodynamic limit) |
| $\mathbb{Z}_2$ symmetry | $\prod_i X_i$ commutes with $H$ |

The operator is built with `SparsePauliOp.from_sparse_list`, identical to the production
`HamiltonianBuilder.build()`.


In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

print("qiskit:", __import__("qiskit").__version__)

# Experiment configuration
J = 1.0
H_VALUES = [1.0, 1.2, 1.4]     # near/above the 1D critical point h_c = 1
N_QUBITS = [10, 20]
P_LAYERS = 2                   # set = 2 to run the SAME experiment with bond-resolved p=2
PERIODIC = True               # open boundary conditions

# NOTE on p=2: everything below is p-agnostic. Setting P_LAYERS = 2 rebuilds the
# bond-resolved circuit with (n_edges + N) * 2 parameters and the GNN selector
# (Layer 2) will look for a p=2 chain_1d model in the zoo automatically.

qiskit: 2.2.3


In [2]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

print("qiskit:", __import__("qiskit").__version__)

# Experiment configuration
J = 1.0
H_VALUES = [1.0, 1.2, 1.4]     # near/above the 1D critical point h_c = 1
N_QUBITS = [10 , 20]
P_LAYERS = 2                   # set = 2 to run the SAME experiment with bond-resolved p=2
PERIODIC = True               # open boundary conditions

# NOTE on p=2: everything below is p-agnostic. Setting P_LAYERS = 2 rebuilds the
# bond-resolved circuit with (n_edges + N) * 2 parameters and the GNN selector
# (Layer 2) will look for a p=2 chain_1d model in the zoo automatically.

qiskit: 2.2.3


## 2. Lattice — 1D chain edges

Open chain: bonds $(i, i{+}1)$. This mirrors `generate_chain_1d(n, periodic=False)`.


In [3]:
def generate_chain_1d(n: int, periodic: bool = False) -> list[tuple[int, int]]:
    """Open (or periodic) 1D chain edge list. Matches qmbp_simulation."""
    edges = [(i, i + 1) for i in range(n - 1)]
    if periodic:
        edges.append((n - 1, 0))
    return edges


for n in N_QUBITS:
    e = generate_chain_1d(n, PERIODIC)
    print(f"N={n:2d}: {len(e)} bonds, first={e[:3]}, last={e[-1]}")

N=10: 10 bonds, first=[(0, 1), (1, 2), (2, 3)], last=(9, 0)
N=20: 20 bonds, first=[(0, 1), (1, 2), (2, 3)], last=(19, 0)


## 3. Build the TFIM Hamiltonian

$H = -J \sum Z_iZ_{i+1} - h \sum X_i$ as a `SparsePauliOp`, identical to `HamiltonianBuilder.build()`.


In [4]:
def build_tfim_1d(n: int, h: float, J: float = 1.0, periodic: bool = False) -> SparsePauliOp:
    """Build H = -J sum_{(i,j)} Z_i Z_j - h sum_i X_i as a SparsePauliOp.

    Identical construction to qmbp_simulation.models.hamiltonian.HamiltonianBuilder.build.
    """
    edges = generate_chain_1d(n, periodic)
    terms: list[tuple[str, list[int], complex]] = []
    # ZZ interaction on each bond (ferromagnetic: -J)
    for (i, j) in edges:
        terms.append(("ZZ", [i, j], -J))
    # Transverse field on each site (-h X)
    for site in range(n):
        terms.append(("X", [site], -h))
    H = SparsePauliOp.from_sparse_list(terms, num_qubits=n)
    assert np.allclose((H - H.adjoint()).simplify().coeffs, 0.0), "H not Hermitian"
    return H


for h in H_VALUES:
    H = build_tfim_1d(10, h, J)
    print(f"N=10, h={h}: {len(H)} Pauli terms (expect {10-1} ZZ + 10 X = 19)")

N=10, h=1.0: 19 Pauli terms (expect 9 ZZ + 10 X = 19)
N=10, h=1.2: 19 Pauli terms (expect 9 ZZ + 10 X = 19)
N=10, h=1.4: 19 Pauli terms (expect 9 ZZ + 10 X = 19)


### 3.1 Explicit Pauli-term listing (unambiguous reference for ED/DMRG)


In [5]:
def print_pauli_terms(H: SparsePauliOp, max_terms: int = 40) -> None:
    labels = H.paulis.to_labels()
    coeffs = H.coeffs
    print(f"{len(H)} terms (qubit order: leftmost char = qubit {H.num_qubits-1} ... rightmost = qubit 0)")
    for k, (lab, c) in enumerate(zip(labels, coeffs)):
        if k >= max_terms:
            print(f"  ... ({len(H) - max_terms} more)")
            break
        print(f"  {c.real:+.3f}  {lab}")


print("=== 1D TFIM, N=6, h=1.0 (small illustration) ===")
print_pauli_terms(build_tfim_1d(6, 1.0, J))

=== 1D TFIM, N=6, h=1.0 (small illustration) ===
11 terms (qubit order: leftmost char = qubit 5 ... rightmost = qubit 0)
  -1.000  IIIIZZ
  -1.000  IIIZZI
  -1.000  IIZZII
  -1.000  IZZIII
  -1.000  ZZIIII
  -1.000  IIIIIX
  -1.000  IIIIXI
  -1.000  IIIXII
  -1.000  IIXIII
  -1.000  IXIIII
  -1.000  XIIIII


## 4. The HVA circuit

Two parametrizations are used in this project:

**(a) Global HVA** (`HVACircuitBuilder.create`): one $\theta_{zz}$ for all bonds, one $\theta_x$ for all
sites → $2p$ parameters.

**(b) Bond-resolved HVA** (`HVACircuitBuilder.create_bond_resolved`): an independent $\theta_{zz}$ per
bond and $\theta_x$ per site → $(n_\text{edges}+N)\cdot p$ parameters. **This is what the GNN zoo
predicts** (Layer 2). Same gate count / depth as global HVA, only more parameters.

Common conventions (both variants):

- **Initial state:** $|+\rangle^{\otimes N}$ (a Hadamard on every qubit).
- **Per layer $\ell$:** `RZZ(2 theta_zz)` on each chain bond, then `RX(2 theta_x)` on each qubit.
- **Gate convention:** the physical factor of 2, i.e. `rzz(2*theta)` = $e^{-i\,\theta\,Z_iZ_j}$ and
  `rx(2*theta)` = $e^{-i\,\theta X}$.


In [6]:
def build_hva_tfim_1d(n: int, p_layers: int, periodic: bool = False):
    """Build the GLOBAL HVA circuit for a 1D TFIM chain (2*p params).

    Matches HVACircuitBuilder.create: |+>^N, then per layer RZZ(2*theta_zz) on
    edges + RX(2*theta_x) on all qubits. 2*p params, [theta_zz_l, theta_x_l].
    """
    edges = generate_chain_1d(n, periodic)
    if not edges:
        raise ValueError(f"No edges for N={n}")
    qc = QuantumCircuit(n)
    theta = ParameterVector("theta", 2 * p_layers)
    qc.h(range(n))
    for layer in range(p_layers):
        theta_zz, theta_x = theta[layer * 2], theta[layer * 2 + 1]
        for (i, j) in edges:
            qc.rzz(2 * theta_zz, i, j)
        for i in range(n):
            qc.rx(2 * theta_x, i)
    print(f"  theta (symbolic, {len(theta)} params): {[str(t) for t in theta]}")
    return qc, theta


def build_hva_bond_resolved_1d(n: int, p_layers: int, periodic: bool = False):
    """Build the BOND-RESOLVED HVA circuit ((n_edges + n) * p params).

    Matches HVACircuitBuilder.create_bond_resolved. Parameter ordering per layer:
    [theta_zz_0..theta_zz_{E-1}, theta_x_0..theta_x_{N-1}]. This is what the GNN predicts.
    """
    edges = generate_chain_1d(n, periodic)
    if not edges:
        raise ValueError(f"No edges for N={n}")
    n_edges = len(edges)
    params_per_layer = n_edges + n
    qc = QuantumCircuit(n)
    theta = ParameterVector("theta", params_per_layer * p_layers)
    qc.h(range(n))
    for layer in range(p_layers):
        offset = layer * params_per_layer
        for k, (i, j) in enumerate(edges):
            qc.rzz(2 * theta[offset + k], i, j)
        for i in range(n):
            qc.rx(2 * theta[offset + n_edges + i], i)
    return qc, theta


qc10, _ = build_hva_tfim_1d(10, P_LAYERS)
print(f"GLOBAL       N=10, p={P_LAYERS}: {qc10.num_parameters} params, depth={qc10.depth()}")
qcbr10, _ = build_hva_bond_resolved_1d(10, P_LAYERS)
_e10 = len(generate_chain_1d(10, PERIODIC))
print(f"BOND-RESOLVED N=10, p={P_LAYERS}: {qcbr10.num_parameters} params "
      f"(expect ({_e10} edges + 10 sites)*{P_LAYERS} = {(_e10 + 10) * P_LAYERS}), "
      f"depth={qcbr10.depth()}")
qc10.draw("text", fold=120)

  theta (symbolic, 4 params): ['theta[0]', 'theta[1]', 'theta[2]', 'theta[3]']
GLOBAL       N=10, p=2: 4 params, depth=14
BOND-RESOLVED N=10, p=2: 38 params (expect (10 edges + 10 sites)*2 = 40), depth=14


┌───┐                 ┌────────────────┐                                    ┌────────────────┐                  »
q_0: ┤ H ├─■───────────────┤ Rx(2*theta[1]) ├───────────────────■────────────────┤ Rx(2*theta[3]) ├──────────────────»
     ├───┤ │ZZ(2*theta[0]) └────────────────┘┌────────────────┐ │ZZ(2*theta[2])  └────────────────┘┌────────────────┐»
q_1: ┤ H ├─■────────────────■────────────────┤ Rx(2*theta[1]) ├─■─────────────────■────────────────┤ Rx(2*theta[3]) ├»
     ├───┤                  │ZZ(2*theta[0])  └────────────────┘┌────────────────┐ │ZZ(2*theta[2])  └────────────────┘»
q_2: ┤ H ├──────────────────■─────────────────■────────────────┤ Rx(2*theta[1]) ├─■─────────────────■────────────────»
     ├───┤                                    │ZZ(2*theta[0])  └────────────────┘┌────────────────┐ │ZZ(2*theta[2])  »
q_3: ┤ H ├────────────────────────────────────■─────────────────■────────────────┤ Rx(2*theta[1]) ├─■────────────────»
     ├───┤                                                      │ZZ(2*theta[0])  └────────────────┘┌────────────────┐»
q_4: ┤ H ├──────────────────────────────────────────────────────■─────────────────■────────────────┤ Rx(2*theta[1]) ├»
     ├───┤                                                                        │ZZ(2*theta[0])  └────────────────┘»
q_5: ┤ H ├────────────────────────────────────────────────────────────────────────■─────────────────■────────────────»
     ├───┤                                                                                          │ZZ(2*theta[0])  »
q_6: ┤ H ├──────────────────────────────────────────────────────────────────────────────────────────■────────────────»
     ├───┤                                                                                                           »
q_7: ┤ H ├───────────────────────────────────────────────────────────────────────────────────────────────────────────»
     ├───┤                                                                                                           »
q_8: ┤ H ├───────────────────────────────────────────────────────────────────────────────────────────────────────────»
     ├───┤                                                                                                           »
q_9: ┤ H ├───────────────────────────────────────────────────────────────────────────────────────────────────────────»
     └───┘                                                                                                           »
«                                                                                                                 »
«q_0: ────────────────────────────────────────────────────────────────────────────────────────────────────────────»
«                                                                                                                 »
«q_1: ────────────────────────────────────────────────────────────────────────────────────────────────────────────»
«     ┌────────────────┐                                                                                          »
«q_2: ┤ Rx(2*theta[3]) ├──────────────────────────────────────────────────────────────────────────────────────────»
«     └────────────────┘┌────────────────┐                                                                        »
«q_3: ─■────────────────┤ Rx(2*theta[3]) ├────────────────────────────────────────────────────────────────────────»
«      │ZZ(2*theta[2])  └────────────────┘┌────────────────┐                                                      »
«q_4: ─■─────────────────■────────────────┤ Rx(2*theta[3]) ├──────────────────────────────────────────────────────»
«     ┌────────────────┐ │ZZ(2*theta[2])  └────────────────┘┌────────────────┐                                    »
«q_5: ┤ Rx(2*theta[1]) ├─■─────────────────■────────────────┤ Rx(2*theta[3]) ├────────────────────────────────────»
«     └────────────────┘┌────────────────┐ │ZZ(2*theta[2])  └────────────────┘┌────────────────┐                  

### 4.1 Circuits for N = 10, 20, 30


In [7]:
circuits_global = {}
circuits_bond_resolved = {}
for n in N_QUBITS:
    circuits_global[n] = build_hva_tfim_1d(n, P_LAYERS)
    circuits_bond_resolved[n] = build_hva_bond_resolved_1d(n, P_LAYERS)
    qb = circuits_bond_resolved[n][0]
    print(f"N={n:2d}: global params={circuits_global[n][0].num_parameters:>2}, "
          f"bond-resolved params={qb.num_parameters:>3}, depth={qb.depth()}, "
          f"gates={dict(qb.count_ops())}")

  theta (symbolic, 4 params): ['theta[0]', 'theta[1]', 'theta[2]', 'theta[3]']
N=10: global params= 4, bond-resolved params= 38, depth=14, gates={'rx': 20, 'rzz': 18, 'h': 10}
  theta (symbolic, 4 params): ['theta[0]', 'theta[1]', 'theta[2]', 'theta[3]']
N=20: global params= 4, bond-resolved params= 78, depth=24, gates={'rx': 40, 'rzz': 38, 'h': 20}


## 5. Reconstruction recipe — everything needed to rebuild the filled circuit

This section makes the notebook a **complete hand-off package**: given only the saved angle vector
$\theta$ (from the `.npz` produced in Layer 2) and the specification below, anyone can rebuild the
exact filled circuit **without the project code and without re-running the GNN**.

**Full specification (bond-resolved HVA, 1D open chain):**

1. **Qubits:** $N$, indexed $0 \dots N-1$.
2. **Edges (bonds):** $(i, i{+}1)$ for $i = 0 \dots N-2$ — that is `E = N-1` bonds, in this exact order.
3. **Initial state:** apply a Hadamard `H` to every qubit → $|+\rangle^{\otimes N}$.
4. **Per layer $\ell = 0 \dots p-1$** (parameter offset $= \ell \cdot (E+N)$):
   - For each bond $k=0\dots E-1$ in order: apply `RZZ(2 · theta[offset + k])` on qubits `edges[k]`.
   - For each site $i=0\dots N-1$: apply `RX(2 · theta[offset + E + i])` on qubit $i$.
5. **Parameter vector layout** (length $(E+N)\cdot p$), per layer:
   $[\theta_{zz,0},\dots,\theta_{zz,E-1},\ \theta_{x,0},\dots,\theta_{x,N-1}]$.
6. **Factor of 2:** the `2*` inside `RZZ`/`RX` is intentional; `rzz(2θ)`$=e^{-i\theta ZZ}$, `rx(2θ)`$=e^{-i\theta X}$.

The `.npz` saved in Layer 2 stores, for every `(N, h)`, both the angle vector and this metadata
(`edges`, `param_order`, `p_layers`, `h`, `N`), so it is self-describing. The function below is the
reference reconstructor (pure `numpy` + `qiskit`).

In [8]:
def rebuild_filled_circuit(n: int, theta: np.ndarray, p_layers: int = 1,
                           periodic: bool = False) -> QuantumCircuit:
    """Rebuild the FILLED bond-resolved HVA circuit from a numeric angle vector.

    Standalone (numpy + qiskit only). Follows the reconstruction recipe above.
    theta length must be (n_edges + n) * p_layers.
    """
    edges = generate_chain_1d(n, periodic)
    n_edges = len(edges)
    params_per_layer = n_edges + n
    expected = params_per_layer * p_layers
    theta = np.asarray(theta, dtype=float).flatten()
    if theta.shape[0] != expected:
        raise ValueError(f"theta length {theta.shape[0]} != expected {expected} "
                         f"= (E={n_edges} + N={n}) * p={p_layers}")
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for layer in range(p_layers):
        offset = layer * params_per_layer
        for k, (i, j) in enumerate(edges):
            qc.rzz(2 * float(theta[offset + k]), i, j)
        for i in range(n):
            qc.rx(2 * float(theta[offset + n_edges + i]), i)
    return qc


# Self-check: rebuild from a dummy theta and confirm gate counts / depth.
# Length derives from P_LAYERS so p=1 and p=2 both work: (E + N) * p params.
_n_chk = N_QUBITS[0]
_n_params_chk = (len(generate_chain_1d(_n_chk, PERIODIC)) + _n_chk) * P_LAYERS
_dummy = np.linspace(-0.5, 0.5, _n_params_chk)
_qc = rebuild_filled_circuit(_n_chk, _dummy, P_LAYERS, PERIODIC)
print(f"rebuild self-check N={_n_chk}, p={P_LAYERS}: {_n_params_chk} params bound, "
      f"depth={_qc.depth()}, gates={dict(_qc.count_ops())}")

rebuild self-check N=10, p=2: 40 params bound, depth=23, gates={'rzz': 20, 'rx': 20, 'h': 10}


## 6. Exact references (N=10 dense ED, N=20 sparse Lanczos)

Ground-truth energies + gaps for the ED/DMRG teams to validate against. N=30 reference is computed
in Layer 2 via DMRG (needs the project's TeNPy solver).

- **N=10:** dense diagonalization (`numpy.linalg.eigh`) — full spectrum + exact ground-state vector
  (the vector is reused for the fidelity calculation in Layer 2).
- **N=20:** sparse Lanczos (`scipy.sparse.linalg.eigsh`, `k=2`, `which='SA'`).

**These numbers are the reference $E_0$ your ED/DMRG must reproduce (open BC, J=1).**


In [9]:
import scipy.sparse.linalg as spla

# Ground-truth cache (topology|N|model|h -> {energy, gap, ...}). p-independent.
# We try to read E0/gap from it before doing any ED/Lanczos work.
try:
    from qmbp_simulation.solvers.ground_truth_cache import GroundTruthCache
    _GT_CACHE = GroundTruthCache()
    print(f"GroundTruthCache: {len(_GT_CACHE)} entries available")
except Exception as _e:
    _GT_CACHE = None
    print(f"GroundTruthCache unavailable ({_e}); will compute all references")

GT_MODEL = "tfim"   # physical Hamiltonian key used in the cache (p-independent)
# Methods whose (E0, gap) come from exact diagonalization. A cached entry is
# only reused as-is when its method is one of these; DMRG/analytical-gap entries
# are treated as non-exact and get overwritten by the exact values we compute.
EXACT_METHODS = {"exact_diag", "exact_lanczos"}


def cached_exact_e0_gap(n: int, h: float):
    """Return (E0, gap) only if the cache holds an EXACT-diagonalization entry.

    Returns None on a miss or when the cached gap came from DMRG/analytical
    correction (those gaps differ from exact diagonalization for N>12).
    """
    if _GT_CACHE is None:
        return None
    hit = _GT_CACHE.get("chain_1d", n, GT_MODEL, h)
    if hit is None or hit.get("method") not in EXACT_METHODS:
        return None
    return float(hit["energy"]), float(hit["gap"])


def store_exact(n: int, h: float, e0: float, gap: float, method: str) -> None:
    """Persist an exact (E0, gap) into GroundTruthCache for future experiments."""
    if _GT_CACHE is None:
        return
    _GT_CACHE.put("chain_1d", n, GT_MODEL, h, energy=e0, gap=gap, method=method)
    _GT_CACHE.flush()


def exact_e0_gap_vec_dense(n: int, h: float, J: float = 1.0, periodic: bool = False):
    """Dense ED: return (E0, gap, ground_state_vector). Feasible to ~N=12-14."""
    H = build_tfim_1d(n, h, J, periodic)
    evals, evecs = np.linalg.eigh(H.to_matrix())
    return float(evals[0]), float(evals[1] - evals[0]), np.ascontiguousarray(evecs[:, 0])


def exact_e0_gap_sparse(n: int, h: float, J: float = 1.0, periodic: bool = False):
    """Sparse Lanczos: return (E0, gap). Feasible to N~22."""
    H = build_tfim_1d(n, h, J, periodic)
    evals = np.sort(spla.eigsh(H.to_matrix(sparse=True).tocsr(), k=2, which="SA",
                               return_eigenvectors=False))
    return float(evals[0]), float(evals[1] - evals[0])


exact_refs = {}         # (n, h) -> (E0, gap)
exact_gs_vectors = {}   # (n, h) -> ground-state vector (small N only, for fidelity)

# Size thresholds for the exact reference method (consistent with N_QUBITS):
#   N <= DENSE_MAX_N        -> dense ED (also yields the GS vector for fidelity)
#   DENSE_MAX_N < N <= LANCZOS_MAX_N -> sparse Lanczos (E0, gap only)
#   N > LANCZOS_MAX_N       -> deferred to GroundTruthCache/DMRG in Layer 2
DENSE_MAX_N = 12
LANCZOS_MAX_N = 18   # matches EXACT_GAP_QUBIT_LIMIT; above this, eigsh is unsafe/infeasible

print("\nExact ground-state energy (1D TFIM, open BC, J=1.0)")
print(f"{'N':>3} {'h':>5} {'E0':>16} {'E0/N':>12} {'gap':>10}  source")
for n in N_QUBITS:
    for h in H_VALUES:
        if n <= DENSE_MAX_N:
            # Dense ED runs regardless (fidelity needs the GS vector, not cached).
            _e_dense, _gap_dense, vec = exact_e0_gap_vec_dense(n, h, J)
            exact_gs_vectors[(n, h)] = vec
            hit = cached_exact_e0_gap(n, h)
            if hit is not None:
                e0, gap = hit
                src = "cache(exact)"
            else:
                e0, gap = _e_dense, _gap_dense
                store_exact(n, h, e0, gap, "exact_diag")
                src = "dense->saved"
        elif n <= LANCZOS_MAX_N:
            hit = cached_exact_e0_gap(n, h)
            if hit is not None:
                e0, gap = hit
                src = "cache(exact)"
            else:
                e0, gap = exact_e0_gap_sparse(n, h, J)
                store_exact(n, h, e0, gap, "exact_lanczos")
                src = "sparse->saved"
        else:
            # Large N: no exact diagonalization here. Reuse an exact cached entry
            # if present; otherwise leave it for Layer 2 (GroundTruthCache/DMRG).
            hit = cached_exact_e0_gap(n, h)
            if hit is not None:
                e0, gap = hit
                src = "cache(exact)"
            else:
                print(f"{n:>3} {h:>5.2f} {'(deferred to Layer 2 DMRG cache)':>40}")
                continue
        exact_refs[(n, h)] = (e0, gap)
        print(f"{n:>3} {h:>5.2f} {e0:>16.10f} {e0/n:>12.8f} {gap:>10.6f}  {src}")

GroundTruthCache: 5370 entries available

Exact ground-state energy (1D TFIM, open BC, J=1.0)
  N     h               E0         E0/N        gap  source
 10  1.00   -12.3814899997  -1.23814900   0.298920  cache(exact)
 10  1.20   -13.9473400175  -1.39473400   0.619007  cache(exact)
 10  1.40   -15.6510755321  -1.56510755   0.979801  cache(exact)
 20  1.00   -25.1077971116  -1.25538986   0.153211  cache(exact)
 20  1.20   -28.1434725234  -1.40717363   0.483459  cache(exact)
 20  1.40   -31.5029566733  -1.57514783   0.860914  cache(exact)


## 7. Fill the circuit with the GNN predictor (Layer 2)

Closes the loop with the project's GNN:

1. Load a `chain_1d`, bond-resolved model (configurable — see next cell).
2. Predict $\theta$ for each $h$; **fill the circuit**.
3. Evaluate $\langle H\rangle$ vs exact $E_0$ for **N=10 (statevector), N=20 (MPS), N=30 (MPS)**.
   The N=30 reference $E_0$/gap is computed with **DMRG** (`ClassicalSolver`).
4. Compute ground-state **fidelity for N=10** using the project's best available method
   (`estimate_fidelity_from_primitives` → exact $|\langle E_0|\psi\rangle|^2$ at this size).
5. Save all $\theta$ + reconstruction metadata to an `.npz`; export filled circuits as images.

> **Model choice is deliberately left open.** `MODEL_CHECKPOINT = None` uses
> `load_best_model_for(..., h_regime="critical")`. To pin a checkpoint, set `MODEL_CHECKPOINT`.
> Candidates the scoreboard reports as best near $h\approx1.0$–$1.5$ for chain_1d:
> `unified_tfim_bond_resolved_chain_1d_n10_p1_20260812T003023.pt`,
> `unified_tfim_br_chain_1d_multiN_6+8+10+12+15+16+20+60_p1_v4.pt`,
> `unifMPNN__chain_1d_p1_h_0p5_1p5.pt`. The zoo `pass_rate` may be stale near $h_c$, so the honest
> $|\Delta E|$ / fidelity printed below is the source of truth.

In [10]:
USE_GNN = True                 # set False to keep Layer 1 standalone only
MODEL_CHECKPOINT = None        # None -> load_best_model_for(h_regime='critical'); or a checkpoint filename/path

# Everything derives from the top-level config (N_QUBITS, P_LAYERS, H_VALUES).
# Evaluate <H> at every N in the experiment; statevector for N<=12, MPS above.
GNN_N_EVAL = list(N_QUBITS)
# Ground-state fidelity needs the exact statevector, feasible only for small N.
STATEVECTOR_MAX_N = 12
FIDELITY_N = [n for n in N_QUBITS if n <= STATEVECTOR_MAX_N]
# Output filename encodes p and boundary conditions so open/periodic and
# p=1/p=2 runs don't overwrite each other (e.g. ..._p1_obc.npz / ..._p2_pbc.npz).
_BC_TAG = "pbc" if PERIODIC else "obc"
THETA_OUT = f"tfim_1d_chain_gnn_theta_p{P_LAYERS}_{_BC_TAG}.npz"   # angles + reconstruction metadata
print(f"config: N_QUBITS={N_QUBITS}, p={P_LAYERS}, H_VALUES={H_VALUES}, periodic={PERIODIC}")
print(f"        GNN_N_EVAL={GNN_N_EVAL}, FIDELITY_N={FIDELITY_N}, THETA_OUT={THETA_OUT}")

config: N_QUBITS=[10, 20], p=2, H_VALUES=[1.0, 1.2, 1.4], periodic=True
        GNN_N_EVAL=[10, 20], FIDELITY_N=[10], THETA_OUT=tfim_1d_chain_gnn_theta_p2_pbc.npz


In [11]:
results_rows = []   # collected for the summary table
saved = {}          # npz payload
filled_circuits = {}

if USE_GNN:
    import torch
    from qiskit.quantum_info import Statevector

    from qmbp_simulation.analysis.fidelity import estimate_fidelity_from_primitives
    from qmbp_simulation.circuits.hva import HVACircuitBuilder
    from qmbp_simulation.execution.mps_backend import MPSBackend
    from qmbp_simulation.models.hamiltonian import HamiltonianBuilder, make_lattice
    from qmbp_simulation.predictors.unified_graph import build_unified_bond_resolved_graph
    from qmbp_simulation.solvers.classical import ClassicalSolver

    # ── Load the model (auto or explicit) ────────────────────────────
    if MODEL_CHECKPOINT is None:
        from qmbp_simulation.predictors.model_zoo import load_best_model_for
        model, entry, source = load_best_model_for(
            "chain_1d", model="tfim_bond_resolved", p_layers=P_LAYERS, h_regime="critical"
        )
        print(f"auto-selected [{source}]: {entry.checkpoint_file}")
        print(f"  pass_rate={entry.pass_rate:.0%}  trained h_range={entry.h_range}")
    else:
        from qmbp_simulation.predictors.model_zoo import _resolve_checkpoint_path, _smart_load_checkpoint
        ckpt = _resolve_checkpoint_path(MODEL_CHECKPOINT) or MODEL_CHECKPOINT
        model = _smart_load_checkpoint(str(ckpt))
        print(f"explicit checkpoint: {ckpt}")
    model.eval()

    hb = HamiltonianBuilder()
    builder = HVACircuitBuilder()
    solver = ClassicalSolver()

    # EvalCache: reuse previously computed <H> energies and N=10 fidelities.
    # Keys are theta-dependent (sha256 of the exact vector) and use h:.2f, so a
    # hit only happens for the SAME predicted theta. Model tag matches the zoo
    # ('tfim_bond_resolved'); GT stays under 'tfim' (p-independent, Section 6).
    EVAL_MODEL = "tfim_bond_resolved"
    try:
        from qmbp_simulation.execution.eval_cache import EvalCache
        _EVAL_CACHE = EvalCache(p_layers=P_LAYERS)
        print(f"EvalCache: {len(_EVAL_CACHE)} entries available")
    except Exception as _e:
        _EVAL_CACHE = None
        print(f"EvalCache unavailable ({_e}); energies/fidelities will be computed")

    def predict_theta(n: int, h: float) -> np.ndarray:
        lat = make_lattice("chain_1d", n, J=J, h=h, periodic=PERIODIC)
        graph = build_unified_bond_resolved_graph(lat, h_value=h, p_layers=P_LAYERS,
                                                  include_circuit_nodes=True)
        with torch.no_grad():
            theta = model(graph).numpy().flatten()
        return np.clip(theta, -np.pi, np.pi)

    def unbound_circuit(n: int):
        lat = make_lattice("chain_1d", n, J=J, h=1.0, periodic=PERIODIC)
        return builder.create_bond_resolved(n, P_LAYERS, lat)

    def ref_e0_gap(n: int, h: float):
        """Exact reference for (n, h). Order: in-memory dict -> GT cache -> solve.

        N<=20 references come from Section 6 (already cache-aware). For N=30 we
        defer to GroundTruthCache.get_or_compute, which reads the cache first
        and only runs DMRG on a miss (then persists it).
        """
        if (n, h) in exact_refs:
            return exact_refs[(n, h)]
        if _GT_CACHE is not None:
            e0, gap = _GT_CACHE.get_or_compute(
                "chain_1d", n, GT_MODEL, h, solver=solver
            )
        else:
            lat = make_lattice("chain_1d", n, J=J, h=h, periodic=PERIODIC)
            gt = solver.solve(hb.build(lat), lat, method="dmrg")
            e0, gap = gt.ground_energy, gt.gap
        exact_refs[(n, h)] = (e0, gap)
        return e0, gap

    for n in GNN_N_EVAL:
        for h in H_VALUES:
            theta = predict_theta(n, h)
            qc_unbound, tv = unbound_circuit(n)
            bound = qc_unbound.assign_parameters({tv[i]: float(theta[i]) for i in range(len(theta))})
            filled_circuits[(n, h)] = bound
            saved[f"theta_N{n}_h{h:.2f}"] = theta
            print(f"theta[N={n},h={h:.2f}] ({len(theta)}): "
                  f"{np.array2string(theta, precision=4, separator=', ', max_line_width=200)}")

            H = hb.build(make_lattice("chain_1d", n, J=J, h=h, periodic=PERIODIC))
            e0, gap = ref_e0_gap(n, h)

            # ── Energy <H>: reuse from EvalCache (keyed by exact theta + h),
            #    else compute (statevector for N<=STATEVECTOR_MAX_N, MPS above)
            #    and persist so the (expensive) MPS is paid only once.
            e_gnn, e_src = None, None
            if _EVAL_CACHE is not None:
                _cached_e = _EVAL_CACHE.get(_EVAL_CACHE.make_key(
                    topology="chain_1d", n_qubits=n, h=h, theta=theta,
                    model=EVAL_MODEL, p_layers=P_LAYERS, J=J))
                if _cached_e is not None:
                    e_gnn, e_src = float(_cached_e), "cache"
            if e_gnn is None:
                if n <= STATEVECTOR_MAX_N:
                    e_gnn, e_src = float(Statevector(bound).expectation_value(H).real), "statevector"
                else:
                    e_gnn, e_src = float(MPSBackend(chi_max=64).evaluate(qc_unbound, H, theta)), "MPS"
                if _EVAL_CACHE is not None:
                    _EVAL_CACHE.put(_EVAL_CACHE.make_key(
                        topology="chain_1d", n_qubits=n, h=h, theta=theta,
                        model=EVAL_MODEL, p_layers=P_LAYERS, J=J), e_gnn)
                    _EVAL_CACHE.flush()

            # ── Fidelity (only for FIDELITY_N): reuse from EvalCache, else compute + persist
            fidelity, fid_method = None, "skipped"
            if n in FIDELITY_N:
                if _EVAL_CACHE is not None:
                    fidelity = _EVAL_CACHE.get_fidelity(
                        "chain_1d", n, h, theta, model=EVAL_MODEL, p_layers=P_LAYERS)
                if fidelity is not None:
                    fid_method = "cache"
                else:
                    fres = estimate_fidelity_from_primitives(
                        qc_unbound, theta, H, gap, n,
                        exact_state=exact_gs_vectors.get((n, h)),
                    )
                    fidelity, fid_method = fres["fidelity"], fres["method"]
                    if _EVAL_CACHE is not None and fidelity is not None:
                        _EVAL_CACHE.put_fidelity(
                            "chain_1d", n, h, theta, float(fidelity),
                            model=EVAL_MODEL, p_layers=P_LAYERS)
                        _EVAL_CACHE.flush()

            results_rows.append({
                "N": n, "h": h, "E_GNN": e_gnn, "E0": e0,
                "|dE|": abs(e_gnn - e0), "dE/gap": abs(e_gnn - e0) / gap if gap else float("nan"),
                "fidelity": fidelity, "fid_method": fid_method, "e_src": e_src,
            })

    # ── Save angles + self-describing reconstruction metadata ─────────
    meta = {
        "topology": "chain_1d", "p_layers": P_LAYERS, "periodic": PERIODIC, "J": J,
        "h_values": list(H_VALUES), "n_qubits": list(GNN_N_EVAL),
        "param_order": "[theta_zz_0..theta_zz_{E-1}, theta_x_0..theta_x_{N-1}] per layer",
        "gate_convention": "rzz(2*theta)=exp(-i theta ZZ); rx(2*theta)=exp(-i theta X)",
        "initial_state": "|+>^N (H on every qubit)",
    }
    for n in GNN_N_EVAL:
        saved[f"edges_N{n}"] = np.array(generate_chain_1d(n, PERIODIC))
    saved["metadata_json"] = np.array(str(meta))
    np.savez(THETA_OUT, **saved)
    print(f"\nSaved {len([k for k in saved if k.startswith('theta_')])} theta vectors "
          f"+ metadata -> {THETA_OUT}")

    # ── Reconstructable HVA weights in the CELL OUTPUT ────────────────
    # Everything below is enough to rebuild each filled circuit standalone,
    # honoring PERIODIC (the edge list already includes the (N-1,0) bond when
    # periodic=True). theta layout per layer: [zz per edge, x per site].
    _bc = 'PERIODIC (ring)' if PERIODIC else 'OPEN chain'
    print(f"\n{'='*70}\nHVA WEIGHTS (theta) - reconstruction block  [{_bc}, p={P_LAYERS}]")
    print(f"initial state |+>^N ; gates rzz(2*theta_zz), rx(2*theta_x)")
    for n in GNN_N_EVAL:
        _edges = generate_chain_1d(n, PERIODIC)
        print(f"\n--- N={n} | edges ({len(_edges)}): {_edges}")
        for h in H_VALUES:
            _th = saved.get(f"theta_N{n}_h{h:.2f}")
            if _th is None:
                continue
            _zz = _th[:len(_edges) * P_LAYERS]  # informational split (layer 0 shown by order)
            print(f"  h={h:.2f}  theta ({len(_th)} = ({len(_edges)}+{n})*{P_LAYERS}):")
            print(f"    {np.array2string(np.asarray(_th), precision=6, separator=', ', max_line_width=110)}")
    print(f"\nRebuild any of them with: rebuild_filled_circuit(N, theta, {P_LAYERS}, {PERIODIC})")
    print('='*70)
else:
    print("USE_GNN is False - Layer 2 skipped (notebook stays standalone).")

auto-selected [per_topology]: unified_tfim_br_chain_1d_multiN_4_p2.pt
  pass_rate=100%  trained h_range=(0.5, 2.3)
EvalCache: 465 entries available
theta[N=10,h=1.00] (40): [ 0.2368,  0.2366,  0.2363,  0.236 ,  0.2358,  0.2356,  0.2354,  0.2354,  0.2352,  0.2349, -0.0029, -0.0034, -0.0031, -0.0027, -0.0023, -0.0022, -0.002 , -0.0017, -0.0005, -0.0018,  0.1958,  0.1847,
  0.191 ,  0.1885,  0.1947,  0.1921,  0.1972,  0.1949,  0.1989,  0.1974, -0.2359, -0.2384, -0.2359, -0.2384, -0.2358, -0.2384, -0.2362, -0.2386, -0.2345, -0.2368]
theta[N=10,h=1.20] (40): [ 2.3661e-01,  2.3640e-01,  2.3595e-01,  2.3547e-01,  2.3491e-01,  2.3449e-01,  2.3399e-01,  2.3356e-01,  2.3309e-01,  2.3247e-01, -2.2420e-03, -3.8536e-03, -3.2515e-03, -2.4511e-03, -1.5317e-03,
 -8.5828e-04,  5.0940e-05,  7.9264e-04,  2.5158e-03,  9.2917e-04,  2.0600e-01,  1.8708e-01,  1.9401e-01,  1.9324e-01,  2.0055e-01,  2.0056e-01,  2.0693e-01,  2.0732e-01,  2.1358e-01,  2.1448e-01,
 -2.3749e-01, -2.4015e-01, -2.3766e-01, -2.4007e

### 7.1 Results table

Rendered with pandas for readability. `|dE|` is the absolute energy error vs the exact reference,
`dE/gap` normalizes it by the spectral gap (the project's pass criterion is `dE/gap < 0.05`).
`fidelity` is the exact ground-state fidelity $|\langle E_0|\psi\rangle|^2$ (N=10 only).


In [12]:
if USE_GNN and results_rows:
    # Column spec: (header, key, formatter, width). PASS marks dE/gap < 0.05.
    cols = [
        ("N",         lambda r: str(r["N"]),                                  4),
        ("h",         lambda r: f"{r['h']:.2f}",                             6),
        ("E_GNN",     lambda r: f"{r['E_GNN']:+.5f}",                       13),
        ("E0 (exact)", lambda r: f"{r['E0']:+.5f}",                         13),
        ("|dE|",      lambda r: f"{r['|dE|']:.5f}",                         11),
        ("dE/gap",    lambda r: f"{r['dE/gap']:.5f}",                       10),
        ("E_src",     lambda r: str(r.get("e_src", "?")),                    12),
        ("fidelity",  lambda r: "N/A" if r["fidelity"] is None else f"{r['fidelity']:.4f}", 10),
        ("fid_method", lambda r: str(r["fid_method"]),                      14),
        ("pass",      lambda r: "PASS" if r["dE/gap"] < 0.05 else "fail",    6),
    ]
    sep = "  |  "
    header = sep.join(h.ljust(w) for h, _, w in cols)
    rule = "-" * len(header)

    print(f"GNN-filled bond-resolved HVA - chain_1d, p={P_LAYERS} "
          f"(fidelity for N={FIDELITY_N}; pass = dE/gap < 0.05)")
    print(rule)
    print(header)
    print(rule)
    rows_sorted = sorted(results_rows, key=lambda r: (r["N"], r["h"]))
    prev_n = None
    for r in rows_sorted:
        if prev_n is not None and r["N"] != prev_n:
            print(rule)   # separator line between different N blocks
        print(sep.join(fmt(r).ljust(w) for _, fmt, w in cols))
        prev_n = r["N"]
    print(rule)
else:
    print("No results to tabulate (USE_GNN is False).")

GNN-filled bond-resolved HVA - chain_1d, p=2 (fidelity for N=[10]; pass = dE/gap < 0.05)
------------------------------------------------------------------------------------------------------------------------------------------------
N     |  h       |  E_GNN          |  E0 (exact)     |  |dE|         |  dE/gap      |  E_src         |  fidelity    |  fid_method      |  pass  
------------------------------------------------------------------------------------------------------------------------------------------------
10    |  1.00    |  -0.22980       |  -12.38149      |  12.15169     |  40.65192    |  statevector   |  0.0075      |  exact           |  fail  
10    |  1.20    |  -0.91538       |  -13.94734      |  13.03196     |  21.05301    |  statevector   |  0.0145      |  exact           |  fail  
10    |  1.40    |  +0.06700       |  -15.65108      |  15.71808     |  16.04210    |  statevector   |  0.0055      |  exact           |  fail  
-----------------------------------------

### 7.2 Export the filled circuits as images

Each GNN-filled bond-resolved circuit is drawn with the `mpl` renderer and saved. We request `.jpg`;
if the backend cannot write JPEG (Pillow missing) we fall back to `.png` (visually identical).


In [13]:
if USE_GNN and filled_circuits:
    import matplotlib.pyplot as plt

    def save_circuit_image(qc, path_stem: str) -> str:
        fig = qc.draw("mpl", fold=40, idle_wires=False)
        for ext in ("jpg", "png"):
            try:
                out = f"{path_stem}.{ext}"
                fig.savefig(out, dpi=150, bbox_inches="tight")
                plt.close(fig)
                return out
            except (ValueError, KeyError):
                continue
        plt.close(fig)
        raise RuntimeError("Could not save circuit image in jpg or png")

    prev_n = None
    for (n, h), qc in sorted(filled_circuits.items()):
        if prev_n is not None and n != prev_n:
            print("-" * 48)
        out = save_circuit_image(qc, f"tfim_1d_chain_N{n}_h{h:.2f}_p{P_LAYERS}_filled")
        print(f"N={n:>2}  |  h={h:.2f}  |  saved {out}")
        prev_n = n
else:
    print("USE_GNN is False - no filled circuits to export.")

N=10  |  h=1.00  |  saved tfim_1d_chain_N10_h1.00_p2_filled.jpg
N=10  |  h=1.20  |  saved tfim_1d_chain_N10_h1.20_p2_filled.jpg
N=10  |  h=1.40  |  saved tfim_1d_chain_N10_h1.40_p2_filled.jpg
------------------------------------------------
N=20  |  h=1.00  |  saved tfim_1d_chain_N20_h1.00_p2_filled.jpg
N=20  |  h=1.20  |  saved tfim_1d_chain_N20_h1.20_p2_filled.jpg
N=20  |  h=1.40  |  saved tfim_1d_chain_N20_h1.40_p2_filled.jpg


### 7.3 Verify the reconstruction recipe reproduces the filled circuit

Confidence check for the hand-off: rebuild each filled circuit from the saved `.npz` angles using the
standalone `rebuild_filled_circuit` (Section 5) and confirm it matches the GNN-filled circuit
bit-for-bit (same instructions). This proves the recipe + `.npz` are sufficient to reconstruct.


In [14]:
if USE_GNN and filled_circuits:
    npz = np.load(THETA_OUT, allow_pickle=True)
    all_ok = True
    prev_n = None
    for (n, h), gnn_qc in sorted(filled_circuits.items()):
        if prev_n is not None and n != prev_n:
            print("-" * 60)
        theta = npz[f"theta_N{n}_h{h:.2f}"]
        rebuilt = rebuild_filled_circuit(n, theta, P_LAYERS, PERIODIC)
        # Compare instruction lists (name, qubits, params) — order-sensitive
        match = (rebuilt.count_ops() == gnn_qc.count_ops()
                 and rebuilt.depth() == gnn_qc.depth()
                 and len(rebuilt.data) == len(gnn_qc.data))
        all_ok &= match
        status = "OK" if match else "MISMATCH"
        print(f"N={n:>2}  |  h={h:.2f}  |  {status:<8}  |  depth={rebuilt.depth():>3}  "
              f"|  gates={dict(rebuilt.count_ops())}")
        prev_n = n
    print(f"\nReconstruction from npz {'fully verified' if all_ok else 'FAILED'}.")
else:
    print("USE_GNN is False - nothing to verify.")

N=10  |  h=1.00  |  OK        |  depth= 23  |  gates={'rzz': 20, 'rx': 20, 'h': 10}
N=10  |  h=1.20  |  OK        |  depth= 23  |  gates={'rzz': 20, 'rx': 20, 'h': 10}
N=10  |  h=1.40  |  OK        |  depth= 23  |  gates={'rzz': 20, 'rx': 20, 'h': 10}
------------------------------------------------------------
N=20  |  h=1.00  |  OK        |  depth= 43  |  gates={'rzz': 40, 'rx': 40, 'h': 20}
N=20  |  h=1.20  |  OK        |  depth= 43  |  gates={'rzz': 40, 'rx': 40, 'h': 20}
N=20  |  h=1.40  |  OK        |  depth= 43  |  gates={'rzz': 40, 'rx': 40, 'h': 20}

Reconstruction from npz fully verified.
